In [1]:
!pip install psycopg2-binary


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import urllib.parse
from sqlalchemy import create_engine

user = 'praktikum_student'
password = urllib.parse.quote_plus('Sdf4$2;d-d30pp')
host = 'rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net'
port = 6432
database = 'data-analyst-afisha'

db_url = f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}?sslmode=require'
engine = create_engine(db_url)

In [7]:
query = '''
SELECT
    p.user_id,
    p.device_type_canonical,
    p.order_id,
    p.created_dt_msk::date AS order_dt,
    p.created_ts_msk       AS order_ts,
    p.currency_code,
    p.revenue,
    p.tickets_count,
    (p.created_dt_msk::date
      - LAG(p.created_dt_msk::date) OVER (
          PARTITION BY p.user_id
          ORDER BY p.created_dt_msk
        )
    )::int AS days_since_prev,
    p.event_id,
    e.event_name_code AS event_name,
    p.service_name,
    e.event_type_main,
    r.region_name,
    c.city_name
FROM afisha.purchases AS p
LEFT JOIN afisha.events  AS e ON p.event_id  = e.event_id
LEFT JOIN afisha.city    AS c ON e.city_id   = c.city_id
LEFT JOIN afisha.regions AS r ON c.region_id = r.region_id
WHERE p.device_type_canonical IN ('mobile', 'desktop')
  AND e.event_type_main NOT IN ('фильм')
'''

data = pd.read_sql(query, engine)
engine.dispose()

print('Размер данных:', data.shape)
print('Жанры:', data['event_type_main'].unique())


Размер данных: (290611, 15)
Жанры: <StringArray>
['театр', 'выставки', 'другое', 'стендап', 'концерты', 'спорт', 'ёлки']
Length: 7, dtype: str


In [8]:
data.to_csv('data/orders.csv', index=False)
print('Сохранено в data/orders.csv')

Сохранено в data/orders.csv
